# Étape 1 — Exploration du dataset Rocket League

Objectifs : décrire rapidement les fichiers CSV fournis, vérifier la volumétrie, afficher des en-têtes/échantillons en évitant de charger entièrement les fichiers volumineux, et fournir un exemple d'ingestion avec DuckDB.
# Étape 1 — Exploration du dataset Rocket League

Objectifs : décrire rapidement les fichiers CSV fournis, vérifier la volumétrie, afficher des en-têtes/échantillons en évitant de charger entièrement les fichiers volumineux, et fournir un exemple d'ingestion avec DuckDB.

## 📊 Étape 1 — Exploration du dataset

### Dataset choisi: Rocket League (statistiques in-game)

**Source:** Dataset local inclus dans `Atelier1/Data/`

**Caractéristiques:**
- Jeu vidéo à fortes interactions (voitures + football)
- Statistiques détaillées in-game (boost, mouvement, positionnement, etc.)
- ~199,000 lignes réparties sur 6 fichiers CSV
- Format UTF-8, séparateur virgule

**Fichiers du dataset:**

In [2]:
import pandas as pd
from pathlib import Path
import json

# Chemin vers les données
DATA_DIR = Path('../data/raw')

# Inventaire des fichiers CSV
csv_files = list(DATA_DIR.glob('*.csv'))

print("📁 Fichiers CSV trouvés:\n")
inventory = {}

for csv_file in sorted(csv_files):
    size_mb = csv_file.stat().st_size / (1024 * 1024)
    
    # Compter les lignes rapidement
    with open(csv_file, 'r', encoding='utf-8') as f:
        line_count = sum(1 for _ in f) - 1  # -1 pour l'en-tête
    
    print(f"  • {csv_file.name:30} {size_mb:>8.2f} MB — {line_count:>10,} lignes")
    
    inventory[csv_file.name] = {
        'path': str(csv_file),
        'size_mb': round(size_mb, 2),
        'rows': line_count
    }

print(f"\n✅ Total: {sum(inv['rows'] for inv in inventory.values()):,} lignes")

# Sauvegarder l'inventaire
with open('step1_summary.json', 'w') as f:
    json.dump(inventory, f, indent=2)
    
print("\n📝 Inventaire sauvegardé dans: step1_summary.json")

📁 Fichiers CSV trouvés:

  • games_by_players.csv              78.65 MB —    106,795 lignes
  • games_by_teams.csv                16.87 MB —     35,594 lignes
  • main.csv                          11.95 MB —     18,740 lignes
  • matches_by_players.csv            23.91 MB —     26,176 lignes
  • matches_by_teams.csv               4.79 MB —     10,594 lignes
  • players_db.csv                     0.09 MB —      1,218 lignes

✅ Total: 199,117 lignes

📝 Inventaire sauvegardé dans: step1_summary.json


In [3]:
# Aperçu des données: players_db
print("👤 PLAYERS_DB — Référentiel des joueurs\n")
df_players = pd.read_csv(DATA_DIR / 'players_db.csv', nrows=5)
print(df_players)
print(f"\nColonnes: {list(df_players.columns)}")
print(f"Total joueurs: {len(pd.read_csv(DATA_DIR / 'players_db.csv'))}")

👤 PLAYERS_DB — Référentiel des joueurs

                  player_id                             player_slug  \
0  6284d6ccc437fde7e02d6e7b  https://octane.gg/players/6e7b-yasudon   
1  60c46a3a9fc1a47e5f11199a    https://octane.gg/players/199a-2die4   
2  5f3d8fdd95f40596eae241b7   https://octane.gg/players/41b7-2piece   
3  61daef8bda9d7ca1c7ba3d33    https://octane.gg/players/3d33-3mari   
4  62712944da9d7ca1c7badbfa    https://octane.gg/players/dbfa-47_vt   

  player_tag       player_name player_country  
0  *YASUDON*               NaN             jp  
1      2Die4  David Morgenrood             za  
2     2Piece     Jayden Horton             us  
3      3mari               NaN             sa  
4      47_VT               NaN             jp  

Colonnes: ['player_id', 'player_slug', 'player_tag', 'player_name', 'player_country']
Total joueurs: 1218


In [5]:
# Aperçu des statistiques détaillées par joueur
print("🎮 GAMES_BY_PLAYERS — Stats in-game détaillées\n")
df_games = pd.read_csv(DATA_DIR / 'games_by_players.csv', nrows=3)
print(f"Nombre total de colonnes: {len(df_games.columns)}")
print(f"\nPremières colonnes: {list(df_games.columns[:10])}")
print(f"\nExemple de 3 premières lignes:")
print(df_games.head(3).to_string())
print(f"\n✅ Métriques détaillées disponibles: boost, mouvement, positionnement, demos, etc.")

🎮 GAMES_BY_PLAYERS — Stats in-game détaillées

Nombre total de colonnes: 103

Premières colonnes: ['game_id', 'color', 'team_id', 'team_region', 'player_id', 'player_tag', 'core_shots', 'core_goals', 'core_saves', 'core_assists']

Exemple de 3 premières lignes:
                    game_id color                   team_id team_region                 player_id player_tag  core_shots  core_goals  core_saves  core_assists  core_score  core_shooting_percentage  boost_bpm  boost_bcpm  boost_avg_amount  boost_amount_collected  boost_amount_stolen  boost_amount_collected_big  boost_amount_stolen_big  boost_amount_collected_small  boost_amount_stolen_small  boost_count_collected_big  boost_count_stolen_big  boost_count_collected_small  boost_count_stolen_small  boost_amount_overfill  boost_amount_overfill_stolen  boost_amount_used_while_supersonic  boost_time_zero_boost  boost_percent_zero_boost  boost_time_full_boost  boost_percent_full_boost  boost_time_boost_0_25  boost_time_boost_25_50  boos

## 🛠️ Étape 2 — Stack Technique

**Architecture :

- **Application:** Jupyter Notebook (Python)
- **Stockage:** DuckDB (base analytique in-process, pas de serveur)
- **Visualisation:** Pandas DataFrames + Matplotlib/Plotly (optionnel)
- **Versionning:** Git / GitHub

**Avantages:**
- ✅ Pas de serveur à installer
- ✅ DuckDB ultra-rapide pour l'analytique
- ✅ Format portable (.duckdb = fichier unique)
- ✅ Compatible notebook web ou local

## 🗄️ Étape 3 — Modèle Relationnel

**Schéma normalisé** créé et validé par le formateur.

**Architecture:**
- **8 dimensions:** Country, Region, Player, Team, Event, Match, Map, Score
- **Tables de stats centrales:** StatMapping (polymorphe), Stat, StatType
- **Pattern polymorphe:** `entity_type` ('player'/'team'/'match') pour flexibilité

**Normalisation:**
- ✅ Tables Country et Region pour éviter doublons géographiques
- ✅ Relations FK entre toutes les dimensions
- ✅ Architecture extensible (nouveaux types de stats faciles à ajouter)

**Fichiers:**
- 📊 Diagramme ER: `docs/schema_final.png`
- 💾 DDL SQL: `sql/schema_final.sql`
- 📝 Source Mermaid: `docs/schema_final.mmd`

duck db ## 📥 Étape 4 — Chargement des Données

Le script `scripts/ingest_duckdb.py` charge les CSV dans DuckDB:
1. Création de la base `data/rl.duckdb` si inexistante
2. Import des 6 fichiers CSV dans des tables
3. Validation automatique (script `scripts/validate_db.py`)

**Vérification du chargement:**

In [13]:
import duckdb

# ⚠️ SOLUTION: DuckDB en mémoire pour démonstration (fichier .duckdb verrouillé)
# Recréer une mini-base en RAM avec les CSV
print("🗄️ DUCKDB - Création base en mémoire pour démonstration\n")
print("=" * 60)

# Connexion DuckDB en mémoire (pas de fichier = pas de verrouillage)
conn = duckdb.connect(':memory:')

print("\n📥 Import des CSV dans DuckDB...")
# Importer les CSV directement dans DuckDB
conn.execute(f"CREATE TABLE players_db AS SELECT * FROM read_csv_auto('{DATA_DIR / 'players_db.csv'}')")
conn.execute(f"CREATE TABLE main AS SELECT * FROM read_csv_auto('{DATA_DIR / 'main.csv'}')")
conn.execute(f"CREATE TABLE games_by_players AS SELECT * FROM read_csv_auto('{DATA_DIR / 'games_by_players.csv'}')")
conn.execute(f"CREATE TABLE games_by_teams AS SELECT * FROM read_csv_auto('{DATA_DIR / 'games_by_teams.csv'}')")
conn.execute(f"CREATE TABLE matches_by_players AS SELECT * FROM read_csv_auto('{DATA_DIR / 'matches_by_players.csv'}')")
conn.execute(f"CREATE TABLE matches_by_teams AS SELECT * FROM read_csv_auto('{DATA_DIR / 'matches_by_teams.csv'}')")

print("✅ Tables créées en mémoire!\n")

# Liste des tables
print("📊 Tables DuckDB:")
tables = conn.execute("SHOW TABLES").df()
print(tables.to_string(index=False))

# Volumétrie
print("\n📈 Nombre de lignes par table (DuckDB):")
print("-" * 60)
for table in ['players_db', 'main', 'games_by_players', 'games_by_teams', 'matches_by_players', 'matches_by_teams']:
    count = conn.execute(f"SELECT COUNT(*) as count FROM {table}").fetchone()[0]
    print(f"  {table:25} {count:>10,} lignes")

# Requête SQL analytique
print("\n🔍 Requête SQL - Top 5 joueurs les plus actifs:")
print("-" * 60)
result = conn.execute("""
    SELECT player_tag, COUNT(*) as games_played
    FROM games_by_players
    GROUP BY player_tag
    ORDER BY games_played DESC
    LIMIT 5
""").df()
print(result.to_string(index=False))

print("\n📊 Requête SQL - Distribution plateformes:")
print("-" * 60)
result2 = conn.execute("""
    SELECT platform, COUNT(*) as count
    FROM games_by_players
    WHERE platform IS NOT NULL
    GROUP BY platform
    ORDER BY count DESC
    LIMIT 5
""").df()
print(result2.to_string(index=False))

conn.close()

print("\n✅ DuckDB fonctionne parfaitement!")
print("\n💡 Note: Base en mémoire (démo) ≈ IDENTIQUE à data/rl.duckdb (fichier verrouillé)")

🗄️ DUCKDB - Création base en mémoire pour démonstration


📥 Import des CSV dans DuckDB...
✅ Tables créées en mémoire!

📊 Tables DuckDB:
              name
  games_by_players
    games_by_teams
              main
matches_by_players
  matches_by_teams
        players_db

📈 Nombre de lignes par table (DuckDB):
------------------------------------------------------------
  players_db                     1,218 lignes
  main                          18,740 lignes
  games_by_players             106,795 lignes
  games_by_teams                35,594 lignes
  matches_by_players            26,176 lignes
  matches_by_teams              10,594 lignes

🔍 Requête SQL - Top 5 joueurs les plus actifs:
------------------------------------------------------------
player_tag  games_played
      Joyo           437
     rise.           437
    Vatira           437
     Ahmad           411
    Atomic           394

📊 Requête SQL - Distribution plateformes:
----------------------------------------------------

## 🗄️ Vérification des données dans la base DuckDB persistante

Contrairement à la démonstration ci-dessus qui utilise une base en mémoire, vérifions maintenant que les données sont bien stockées de manière persistante dans le fichier `data/rl.duckdb`.

In [3]:
import duckdb
from pathlib import Path

# Chemin vers la base DuckDB persistante (remonte d'un niveau depuis notebooks/)
DB_PATH = Path('..') / 'data' / 'rl.duckdb'

print(f"📂 Connexion à la base : {DB_PATH}")
print(f"   Taille du fichier : {DB_PATH.stat().st_size / (1024*1024):.2f} MB\n")

# Connexion en lecture seule à la base persistante
conn = duckdb.connect(str(DB_PATH), read_only=True)

# Liste des tables dans la base
print("=" * 60)
print("📊 TABLES STOCKÉES DANS LA BASE DUCKDB")
print("=" * 60)

tables_info = conn.execute("""
    SELECT 
        table_name,
        (SELECT COUNT(*) FROM information_schema.tables t2 
         WHERE t2.table_name = t.table_name) as exists
    FROM information_schema.tables t
    WHERE table_schema = 'main'
    ORDER BY table_name
""").fetchall()

for table_name, _ in tables_info:
    row_count = conn.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
    print(f"✓ {table_name:<25} {row_count:>10,} lignes")

# Total des lignes
total_rows = sum([conn.execute(f"SELECT COUNT(*) FROM {t[0]}").fetchone()[0] for t in tables_info])
print("=" * 60)
print(f"   TOTAL                      {total_rows:>10,} lignes")
print("=" * 60)

# Vérification d'intégrité : quelques exemples de données
print("\n🔍 Exemples de données stockées :")
print("\n1️⃣  Top 3 joueurs par nombre de matchs :")
top_players = conn.execute("""
    SELECT p.player_name, COUNT(*) as nb_matches
    FROM matches_by_players m
    JOIN players_db p ON m.player_id = p.player_id
    GROUP BY p.player_name
    ORDER BY nb_matches DESC
    LIMIT 3
""").fetchall()

for player, count in top_players:
    print(f"   • {player}: {count} matchs")

print("\n2️⃣  Distribution des plateformes :")
platforms = conn.execute("""
    SELECT 
        COALESCE(platform, 'non spécifié') as platform, 
        COUNT(*) as count
    FROM games_by_players
    GROUP BY platform
    ORDER BY count DESC
    LIMIT 5
""").fetchall()

for platform, count in platforms:
    print(f"   • {platform}: {count:,} entrées")

conn.close()
print("\n✅ Données confirmées : La base DuckDB est opérationnelle et contient toutes les données !")

📂 Connexion à la base : ..\data\rl.duckdb
   Taille du fichier : 49.51 MB

📊 TABLES STOCKÉES DANS LA BASE DUCKDB
✓ games_by_players             106,795 lignes
✓ games_by_teams                35,594 lignes
✓ main                          18,740 lignes
✓ matches_by_players            26,176 lignes
✓ matches_by_teams              10,594 lignes
✓ players_db                     1,218 lignes
   TOTAL                         199,117 lignes

🔍 Exemples de données stockées :

1️⃣  Top 3 joueurs par nombre de matchs :
   • None: 8092 matchs
   • Matheus Rodrigues: 94 matchs
   • Finlay Ferguson: 90 matchs

2️⃣  Distribution des plateformes :
   • steam: 100,674 entrées
   • epic: 3,801 entrées
   • non spécifié: 1,182 entrées
   • ps4: 519 entrées
   • xbox: 498 entrées

✅ Données confirmées : La base DuckDB est opérationnelle et contient toutes les données !
